# Einstein — Unified Physics Hypothesis Explorer
### Interactive Notebook: All Phases Demonstration

| Phase | Topic |
|---|---|
| 2 | Symbolic tensor calculus (geometry) |
| 3 | Scalar field simulations |
| 4 | Geometry visualizations |
| 5 | Hypothesis testing & observational constraints |
| 6 | LLM-assisted consistency analysis & literature mapping |

> **No API key required.** Phase 6 uses `MockLLMClient` by default.  
> Set `LLM_PROVIDER=copilot` + `GITHUB_TOKEN` for GitHub Copilot responses.


## Setup

In [2]:
import sys, os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#0d1117',
    'axes.edgecolor':   '#30363d',
    'grid.color':       '#21262d',
    'text.color':       '#e6edf3',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
})

import pathlib
pathlib.Path('../outputs').mkdir(exist_ok=True)

print("Setup complete.")


Setup complete.


---
## Phase 2 — Symbolic Tensor Calculus

Full GR tensor hierarchy: g_μν → g^μν → Γ^λ_μν → R^ρ_σμν → R_μν → R → G_μν.


In [3]:
import sympy as sp
from einstein.geometry.manifold import Manifold

# Minkowski (flat spacetime)
coords_mink = list(sp.symbols('t x y z'))
eta = sp.Matrix([[-1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1]])
mink = Manifold(coords_mink, eta, name='Minkowski')
print(f"Minkowski  R = {mink.ricci_scalar}   (expected: 0)")
print(f"           is_flat = {mink.is_flat()}")


Minkowski  R = 0   (expected: 0)
           is_flat = True


In [4]:
# Schwarzschild (vacuum, spherical symmetry)
r, theta = sp.Symbol('r', positive=True), sp.Symbol('theta', positive=True)
t_s, phi  = sp.symbols('t phi')
M, G, c   = sp.symbols('M G c', positive=True)

f = 1 - 2*G*M / (c**2 * r)
g_schw = sp.Matrix([
    [-f*c**2, 0,   0,  0],
    [0,       1/f, 0,  0],
    [0,       0,   r**2, 0],
    [0,       0,   0,  r**2*sp.sin(theta)**2],
])
schw = Manifold([t_s, r, theta, phi], g_schw, name='Schwarzschild')
R_schw = sp.simplify(schw.ricci_scalar)
print(f"Schwarzschild  R = {R_schw}   (expected: 0 — vacuum solution)")


Schwarzschild  R = 0   (expected: 0 — vacuum solution)


In [5]:
# 2-Sphere S²
theta2, phi2 = sp.symbols('theta phi', positive=True)
a = sp.Symbol('a', positive=True)
g_s2 = sp.Matrix([[a**2, 0], [0, a**2 * sp.sin(theta2)**2]])
s2 = Manifold([theta2, phi2], g_s2, name='S2')
R_s2 = sp.simplify(s2.ricci_scalar)
print(f"S²  R = {R_s2}   (expected: 2/a²)")
print(f"\nMetric LaTeX:\n{s2.latex_metric()}")


S²  R = 2/a**2   (expected: 2/a²)

Metric LaTeX:
\left[\begin{matrix}a^{2} & 0\\0 & a^{2} \sin^{2}{\left(\theta \right)}\end{matrix}\right]


---
## Phase 3 — Scalar Field Simulations

Leapfrog (Störmer-Verlet) integrator — symplectic, energy-conserving.

Available simulators:
- `simulate_kg_1d` — Klein-Gordon 1D, flat spacetime
- `simulate_kg_2d` — Klein-Gordon 2D, flat spacetime
- `simulate_phi4_1d` — φ⁴ kink solitons (H3: matter as excitation)
- `simulate_kg_curved_1d` — KG on Schwarzschild background


In [6]:
from einstein.simulation.scalar_field import (
    simulate_kg_1d,
    simulate_kg_2d,
    simulate_phi4_1d,
    simulate_kg_curved_1d,
)

# Klein-Gordon 1D: Gaussian pulse propagation
def gaussian_phi(x): return np.exp(-x**2)
def zero_pi(x): return np.zeros_like(x)

res_kg = simulate_kg_1d(
    n_x=200, x_max=10.0, t_max=3.0, dt=0.01, mass=1.0,
    initial_phi=gaussian_phi, initial_pi=zero_pi,
    boundary='absorbing', save_every=5,
)

x_kg = np.linspace(-10, 10, 200)
n_frames = len(res_kg.phi)
t0_idx  = 0
t15_idx = n_frames // 2
t30_idx = n_frames - 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_kg, res_kg.phi[t0_idx],  color='#58a6ff', label=f't={res_kg.times[t0_idx]:.1f}')
axes[0].plot(x_kg, res_kg.phi[t15_idx], color='#3fb950', label=f't={res_kg.times[t15_idx]:.1f}')
axes[0].plot(x_kg, res_kg.phi[t30_idx], color='#f78166', label=f't={res_kg.times[t30_idx]:.1f}')
axes[0].set_title('Klein-Gordon 1D — Gaussian Pulse'); axes[0].legend()
axes[0].set_xlabel('x'); axes[0].set_ylabel('φ(x,t)')

axes[1].plot(res_kg.times, res_kg.energy, color='#e3b341')
axes[1].set_title('Energy (absorbing BC → decay)'); axes[1].set_xlabel('t'); axes[1].set_ylabel('E(t)')

plt.tight_layout()
plt.savefig('../outputs/phase3_kg1d.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Frames saved: {n_frames}")


Frames saved: 61


In [7]:
# φ⁴ Kink Soliton — H3: matter as stable field excitation
res_kink = simulate_phi4_1d(
    n_x=600, x_max=15.0, t_max=5.0, dt=0.005,
    lam=1.0, v_vev=1.0, n_kinks=1,
    save_every=20,
)

x_k = np.linspace(-15, 15, 600)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_k, res_kink.phi[0], color='#58a6ff', lw=2)
axes[0].axhline( 1, color='#f78166', ls='--', alpha=0.5, label='v = ±1')
axes[0].axhline(-1, color='#f78166', ls='--', alpha=0.5)
axes[0].set_title('φ⁴ Kink Soliton — H3: Matter as Field Excitation')
axes[0].set_xlabel('x'); axes[0].set_ylabel('φ(x)'); axes[0].legend()

axes[1].plot(res_kink.times, res_kink.energy, color='#e3b341')
axes[1].set_title('Kink Energy Stability'); axes[1].set_xlabel('t'); axes[1].set_ylabel('E(t)')

plt.tight_layout()
plt.savefig('../outputs/phase3_kink.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Kink static energy: {res_kink.energy[0]:.4f}  (theory: 4v³√(λ/2)/3 ≈ 0.9428)")


Kink static energy: 0.9427  (theory: 4v³√(λ/2)/3 ≈ 0.9428)


In [8]:
# KG on Schwarzschild — quasinormal ringing & energy decay
res_schw = simulate_kg_curved_1d(
    n_x=300, r_min=0.5, r_max=30.0, t_max=20.0, dt=0.05,
    mass=0.0, r_s=1.0, l_angular=0, save_every=4,
)

mid = res_schw.phi.shape[1] // 2
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(res_schw.times, res_schw.phi[:, mid], color='#58a6ff', lw=1)
axes[0].set_title('KG on Schwarzschild (l=0) — Quasinormal Ringing')
axes[0].set_xlabel('t'); axes[0].set_ylabel('φ(r_mid, t)')

axes[1].plot(res_schw.times, res_schw.energy, color='#e3b341')
axes[1].set_title('Energy Decay through Horizon'); axes[1].set_xlabel('t'); axes[1].set_ylabel('E(t)')

plt.tight_layout()
plt.savefig('../outputs/phase3_schwarzschild.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Phase 4 — Geometry Visualizations


In [9]:
from einstein.visualization.geometry import (
    plot_embedding_diagram,
    plot_curvature_heatmap,
    plot_schwarzschild_spacetime,
    plot_expansion,
    plot_field_excitation,
)
import numpy as np

# Schwarzschild embedding: z = sqrt(8*r_s*(r - r_s)), r_s=1
def schw_surface(u, v):
    r = np.sqrt(u**2 + v**2) + 1.01
    return np.sqrt(np.maximum(8.0 * (r - 1.0), 0))

fig = plot_embedding_diagram(schw_surface, u_range=(-4,4), v_range=(-4,4),
                              title='Schwarzschild Embedding Diagram')
plt.savefig('../outputs/phase4_embedding.png', dpi=120, bbox_inches='tight')
plt.show()


In [10]:
# Ricci scalar heatmap for Schwarzschild-like curvature
def ricci_fn(x, y):
    r2 = x**2 + y**2 + 0.1
    return -2.0 / r2**2

fig = plot_curvature_heatmap(ricci_fn, title='Ricci Scalar Curvature (Schwarzschild-like)')
plt.savefig('../outputs/phase4_ricci.png', dpi=120, bbox_inches='tight')
plt.show()

fig = plot_schwarzschild_spacetime(r_s=1.0)
plt.savefig('../outputs/phase4_lightcones.png', dpi=120, bbox_inches='tight')
plt.show()

fig = plot_expansion()
plt.savefig('../outputs/phase4_cosmo.png', dpi=120, bbox_inches='tight')
plt.show()

fig = plot_field_excitation(title='H3: φ⁴ Kink — Matter as Excitation')
plt.savefig('../outputs/phase4_excitation.png', dpi=120, bbox_inches='tight')
plt.show()

print("All Phase 4 visualizations saved to outputs/")


All Phase 4 visualizations saved to outputs/


---
## Phase 5 — Hypothesis Testing

- 14 computable numerical predictions  
- 19 cross-consistency checks (vs GR / QFT / SR / QM / ΛCDM)  
- 11 observational constraints (CODATA 2018, Planck 2018)


In [11]:
from einstein.hypotheses import build_registry

registry = build_registry()

print("H1–H6 Hypothesis Overview")
print("=" * 65)
for h in registry.all():
    print(f"  {h.id}  [{h.status.value.upper():12s}]  {h.name}")


H1–H6 Hypothesis Overview
  H1  [SUPPORTED   ]  Space is an active medium (quantum vacuum)
  H2  [SUPPORTED   ]  Space and energy are manifestations of the same fundamental structure
  H3  [SUPPORTED   ]  Matter is a stable local excitation of space
  H4  [SUPPORTED   ]  Gravity is emergent, not fundamental
  H5  [SUPPORTED   ]  Time is emergent
  H6  [SUPPORTED   ]  The expansion of the universe is a dynamic transformation of the fundamental structure


In [12]:
from einstein.hypotheses import predictions as pred

# All prediction functions
pred_fns = [
    ('H1-P1', 'Vacuum energy density ratio', pred.h1_p1_vacuum_energy_density),
    ('H1-P2', 'Zero-point energy harmonic osc.', pred.h1_p2_zero_point_energy_harmonic_oscillator),
    ('H1-P3', 'Casimir force sign', pred.h1_p3_casimir_force_sign),
    ('H2-P1', 'Natural units dimensional consistency', pred.h2_p1_natural_units_dimensional_consistency),
    ('H2-P2', 'Energy-length trade-off', pred.h2_p2_energy_length_trade_off),
    ('H3-P1', 'Kink mass formula', pred.h3_p1_kink_mass_formula),
    ('H3-P2', 'Kink width', pred.h3_p2_kink_width),
    ('H3-P3', 'Kink topological charge', pred.h3_p3_kink_topological_charge),
    ('H3-P4', 'Kink potential minima', pred.h3_p4_kink_potential_minima),
    ('H4-P1', 'Verlinde Newton force', pred.h4_p1_verlinde_newton_force),
    ('H4-P2', 'Gravitational entropy area law', pred.h4_p2_gravitational_entropy_area_law),
    ('H5-P1', 'Time dilation Schwarzschild', pred.h5_p1_time_dilation_schwarzschild),
    ('H5-P2', 'Planck time minimum interval', pred.h5_p2_planck_time_as_minimum_interval),
    ('H5-P3', 'SR time dilation', pred.h5_p3_special_relativistic_time_dilation),
    ('H6-P1', 'Hubble sphere radius', pred.h6_p1_hubble_sphere_radius),
    ('H6-P2', 'Scale factor deceleration→accel.', pred.h6_p2_scale_factor_deceleration_to_acceleration),
    ('H6-P3', 'Comoving distance integral', pred.h6_p3_comoving_distance_integral),
]

print(f"{'ID':<8} {'Name':<38} {'Pass'}")
print("-" * 55)
for pid, name, fn in pred_fns:
    passed, computed, expected, note = fn()
    tag = '✓' if passed else '✗'
    print(f"  {tag}  {pid:<6} {name:<38}")


ID       Name                                   Pass
-------------------------------------------------------
  ✓  H1-P1  Vacuum energy density ratio           
  ✓  H1-P2  Zero-point energy harmonic osc.       
  ✓  H1-P3  Casimir force sign                    
  ✓  H2-P1  Natural units dimensional consistency 
  ✓  H2-P2  Energy-length trade-off               
  ✓  H3-P1  Kink mass formula                     
  ✓  H3-P2  Kink width                            
  ✓  H3-P3  Kink topological charge               
  ✓  H3-P4  Kink potential minima                 
  ✓  H4-P1  Verlinde Newton force                 
  ✓  H4-P2  Gravitational entropy area law        
  ✓  H5-P1  Time dilation Schwarzschild           
  ✓  H5-P2  Planck time minimum interval          
  ✓  H5-P3  SR time dilation                      
  ✓  H6-P1  Hubble sphere radius                  
  ✓  H6-P2  Scale factor deceleration→accel.      
  ✓  H6-P3  Comoving distance integral            


In [13]:
from einstein.hypotheses.consistency import ConsistencyChecker
from collections import Counter

checker = ConsistencyChecker()
results = checker.run_all()

counts = Counter(r.verdict.value for r in results)
print("Cross-consistency summary (19 checks vs GR/QFT/SR/QM/ΛCDM):")
for verdict, n in sorted(counts.items()):
    print(f"  {verdict:<14} {'█' * n} {n}")


Cross-consistency summary (19 checks vs GR/QFT/SR/QM/ΛCDM):
  consistent     █████████████ 13
  subsumes       █ 1
  tension        █████ 5


In [14]:
from einstein.hypotheses.observational import ObservationalChecker

obs = ObservationalChecker()
obs_results = obs.run_all()

passed = sum(1 for r in obs_results if r.passed)
print(f"Observational constraints: {passed}/{len(obs_results)} passed\n")
for r in obs_results:
    tag = '✓' if r.passed else '✗'
    print(f"  {tag}  {r.constraint_name}")


Observational constraints: 7/7 passed

  ✓  Casimir pressure (d=1 μm)
  ✓  Cosmological constant
  ✓  Hubble constant
  ✓  Deceleration-acceleration transition redshift
  ✓  Electron mass
  ✓  Planck time
  ✓  Bekenstein-Hawking entropy (M_sun BH)


---
## Phase 6 — LLM-Assisted Analysis

`MockLLMClient` is used by default (deterministic, no API key).  
To switch to GitHub Copilot: `os.environ['LLM_PROVIDER'] = 'copilot'` and set `GITHUB_TOKEN`.


In [15]:
from einstein.hypotheses import build_registry
from einstein.llm import get_client, LLMConsistencyChecker, LiteratureMapper

registry = build_registry()
client = get_client()   # reads LLM_PROVIDER env var, defaults to 'mock'
print(f"Provider : {client.provider_name}")
print(f"Model    : {client.model_name}")


Provider : mock
Model    : mock-v1


In [16]:
# Pairwise consistency matrix (15 hypothesis pairs)
llm_checker = LLMConsistencyChecker(client, registry)
summary = llm_checker.summary()

print("LLM Pairwise Consistency (15 pairs):")
for verdict, n in sorted(summary.items()):
    print(f"  {verdict:<14} {'█' * n} {n}")


LLM Pairwise Consistency (15 pairs):
  consistent     █████████████ 13
  tension        ██ 2


In [17]:
import matplotlib.colors as mcolors

ids = ['H1','H2','H3','H4','H5','H6']
verdict_score = {'consistent': 1, 'tension': 0, 'conflict': -1}
mat_dict = llm_checker.matrix_as_dict()

grid = np.zeros((6, 6))
for i, h1 in enumerate(ids):
    for j, h2 in enumerate(ids):
        if h1 == h2:
            grid[i, j] = 1.0
        else:
            grid[i, j] = verdict_score.get(mat_dict.get(h1, {}).get(h2, 'tension'), 0)

cmap = mcolors.LinearSegmentedColormap.from_list('rg', ['#f78166','#e3b341','#3fb950'], N=256)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(grid, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(ids); ax.set_yticklabels(ids)
ax.set_title('LLM Pairwise Consistency Matrix\n(green=consistent, yellow=tension, red=conflict)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('../outputs/phase6_consistency_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


In [18]:
# Validate a new hypothesis
new_stmt = (
    "Dark matter consists of stable φ⁶ solitons of a new scalar field "
    "weakly coupled to the Standard Model through a portal interaction."
)
validation = llm_checker.validate_new_hypothesis(new_stmt)

print(f"Statement  : {validation.statement[:80]}...")
print(f"Verdict    : {validation.verdict}")
print(f"Novelty    : {validation.novelty}")
print(f"Confidence : {validation.confidence:.2f}")
print(f"Consistent with: {validation.consistent_with}")
if validation.suggestions:
    print("Suggestions:")
    for s in validation.suggestions:
        print(f"  - {s}")


Statement  : Dark matter consists of stable φ⁶ solitons of a new scalar field weakly coupled ...
Verdict    : partial
Novelty    : complementary
Confidence : 0.60
Consistent with: ['H3', 'H1']
Suggestions:
  - Specify the field parameters connecting to the Standard Model mass scale.
  - Check whether the proposed soliton produces an entropy gradient consistent with H4.


In [19]:
# Literature mapping
mapper = LiteratureMapper(client, registry)
all_mappings = mapper.map_all()

print("Theory mappings per hypothesis:")
print("-" * 55)
for hid, mapping in all_mappings.items():
    theories = [t.name for t in mapping.theories]
    print(f"  {hid}: {', '.join(theories[:3])}{'...' if len(theories) > 3 else ''}")


Theory mappings per hypothesis:
-------------------------------------------------------
  H1: Quantum Field Theory (Weinberg), Casimir Effect (Casimir 1948), Cosmological Constant Problem (Weinberg 1989)
  H2: Loop Quantum Gravity (Rovelli), Causal Dynamical Triangulations, Einstein Field Equations
  H3: Soliton Theory (Rajaraman 1982), Standard Model QFT, Skyrme Model (Skyrme 1961)...
  H4: Entropic Gravity (Verlinde 2011), Thermodynamics of Spacetime (Jacobson 1995), AdS/CFT Correspondence...
  H5: Page-Wootters Mechanism (1983), Relational Quantum Mechanics (Rovelli), Causal Sets (Bombelli et al. 1987)...
  H6: FLRW Cosmology, Inflationary Cosmology (Guth 1981), Dark Energy / Λ (Planck 2018)...


In [20]:
# Bibliography (BibTeX) for H4
bib = mapper.generate_bibliography('H4')
print(f"Bibliography for H4 — {len(bib.entries)} entries\n")
print(bib.to_bibtex()[:500] + "...")


Bibliography for H4 — 4 entries

@article{Verlinde2011,
  author  = {Verlinde, E.},
  year    = {2011},
  title   = {On the Origin of Gravity and the Laws of Newton},
  journal = {JHEP},
  volume  = {04},
  pages   = {029},
  doi     = {10.1007/JHEP04(2011)029},
}

@article{Jacobson1995,
  author  = {Jacobson, T.},
  year    = {1995},
  title   = {Thermodynamics of Spacetime: The Einstein Equation of State},
  journal = {Phys. Rev. Lett.},
  volume  = {75},
  pages   = {1260},
  doi     = {10.1103/PhysRevLett.75.1260},
}

@arti...


---
## End-to-End Report Generation

Generates the full Markdown report (all phases + LLM section) and a BibTeX bibliography.


In [21]:
from einstein.hypotheses.report import generate_report

report_md = generate_report(
    include_llm=True,
    llm_client=client,
    output_path='../outputs/hypothesis_report_phase7.md',
    bibtex_path='../outputs/references.bib',
)

lines = report_md.splitlines()
print(f"Report generated: {len(lines)} lines")
print(f"Saved to: outputs/hypothesis_report_phase7.md")
print(f"BibTeX  : outputs/references.bib")
print()
print('\n'.join(lines[:20]))
print("...")


Report generated: 400 lines
Saved to: outputs/hypothesis_report_phase7.md
BibTeX  : outputs/references.bib

# Einstein — Hypothesis Testing Report

> Generated: 2026-05-21 21:19:02

---

## Overview

This report evaluates the six core hypotheses H1–H6 against:
1. Derived numerical predictions (analytic and computational)
2. Cross-consistency with established theories (GR, QFT, SR, QM, ΛCDM)
3. Observational / experimental constraints

## Hypothesis Status Summary

| ID | Name | Status | Predictions | Computable | Theory relations |
|---|---|---|---|---|---|
| **H1** | Space is an active medium (quantum vacuum) | ✅ supported | 3 | 3 | 4 |
| **H2** | Space and energy are manifestations of the same fundamental structure | ✅ supported | 2 | 2 | 4 |
| **H3** | Matter is a stable local excitation of space | ✅ supported | 4 | 4 | 4 |
...


---
## Test Suite


In [22]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pytest', '../tests/', '-q', '--tb=short'],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-1000:])


........................................................................ [ 32%]
........................................................................ [ 64%]
........................................................................ [ 96%]
........                                                                 [100%]
224 passed in 1.65s



---
## Summary

| Phase | Module | Tests |
|---|---|---|
| 2 | `einstein.geometry.manifold` | 18 |
| 3 | `einstein.simulation.scalar_field` | 28 |
| 4 | `einstein.visualization.geometry` | 21 |
| 5 | `einstein.hypotheses.*` | 68 |
| 6 | `einstein.llm.*` | 89 |
| **Total** | | **224** |

All outputs saved to `outputs/`.
